In [1]:

import optuna
from sklearn.model_selection import train_test_split, cross_val_score
from lightgbm import LGBMClassifier, LGBMRegressor
import polars as pl 
from Model_Building.ModelBuilder import ModelBuilder
from params.targets import NIL_TARGET, TARGET_COL
from params.generic_params import DO_NOT_MODEL
from Model_Building.rfe_cv import run_rfe
import lightgbm as lgb

/Users/johnwragg/Desktop/DS_work/Hastings_Solution/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Setup model run
train = pl.read_parquet(
    "/Users/johnwragg/Desktop/DS_work/Hastings_Solution/Prepped_Data/train.parquet"
)

test = pl.read_parquet(
    "/Users/johnwragg/Desktop/DS_work/Hastings_Solution/Prepped_Data/test.parquet"
)

features = [col for col in train.columns if col not in DO_NOT_MODEL]

# run rfe for each model
feature_selection_dict = {}
for model in ["nil", "sev"]:
    target = NIL_TARGET if model == "nil" else TARGET_COL
    model_features = run_rfe(train, features, target, model)
    feature_selection_dict[model] = model_features



Optimal number of features: 24
Optimal number of features: 26


In [3]:
# For nil model
nil_features = feature_selection_dict["nil"]

y_train_nil = train[NIL_TARGET].to_numpy()
X_train_nil = train.select(nil_features).to_pandas()

year_idx_nil = nil_features.index("loss_year")
others_nil = [i for i in range(len(nil_features)) if i != year_idx_nil]
nil_interaction_constraints = [[year_idx_nil], others_nil]  # year in own interaction group


def objective(trial):
    n_estimators = trial.suggest_int("n_estimators", 10, 200)
    max_depth = trial.suggest_int("max_depth", 2, 32, log=True)
    min_child_samples = trial.suggest_int("min_child_samples", 5, 100)

    clf = LGBMClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_child_samples=min_child_samples,
        random_state=42,
        interaction_constraints=nil_interaction_constraints,
        objective="binary",
        verbose=-1,
    )

    score = cross_val_score(clf, X_train_nil, y_train_nil, cv=3, scoring="accuracy").mean()
    return score


In [4]:

study = optuna.create_study(direction="maximize")

[I 2026-11-07 19:53:49,163] A new study created in memory with name: no-name-b14eaa8d-c87b-4325-9d7e-ee3fc9460786


In [5]:
study.optimize(objective, n_trials=5)

[I 2026-11-07 19:53:49,666] Trial 0 finished with value: 0.7672554792280013 and parameters: {'n_estimators': 86, 'max_depth': 3, 'min_child_samples': 87}. Best is trial 0 with value: 0.7672554792280013.
[I 2026-11-07 19:53:51,066] Trial 1 finished with value: 0.7579326136735361 and parameters: {'n_estimators': 133, 'max_depth': 28, 'min_child_samples': 88}. Best is trial 0 with value: 0.7672554792280013.
[I 2026-11-07 19:53:53,052] Trial 2 finished with value: 0.7512266928361138 and parameters: {'n_estimators': 198, 'max_depth': 32, 'min_child_samples': 97}. Best is trial 0 with value: 0.7672554792280013.
[I 2026-11-07 19:53:54,146] Trial 3 finished with value: 0.7587504088976121 and parameters: {'n_estimators': 167, 'max_depth': 7, 'min_child_samples': 98}. Best is trial 0 with value: 0.7672554792280013.
[I 2026-11-07 19:53:54,315] Trial 4 finished with value: 0.7708537782139353 and parameters: {'n_estimators': 12, 'max_depth': 9, 'min_child_samples': 64}. Best is trial 4 with value: 

In [6]:

print("Best hyperparameters:", study.best_params)
print("Best accuracy:", study.best_value)

Best hyperparameters: {'n_estimators': 12, 'max_depth': 9, 'min_child_samples': 64}
Best accuracy: 0.7708537782139353


In [7]:
# For sev model
sev_features = feature_selection_dict["sev"]
train_sev = train.filter(pl.col(NIL_TARGET) == 1)
test_sev = test.filter(pl.col(NIL_TARGET) == 1)

y_train_sev = train_sev[TARGET_COL].to_numpy()
X_train_sev = train_sev.select(sev_features).to_pandas()

year_idx = sev_features.index("loss_year")
others = [i for i in range(len(sev_features)) if i != year_idx]
sev_interaction_constraints = [[year_idx], others]  # year in own interaction group


def objective_sev(trial):
    n_estimators = trial.suggest_int("n_estimators", 10, 200)
    max_depth = trial.suggest_int("max_depth", 2, 32, log=True)
    min_child_samples = trial.suggest_int("min_child_samples", 5, 100)

    reg = LGBMRegressor(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_child_samples=min_child_samples,
        random_state=42,
        interaction_constraints=sev_interaction_constraints,
        objective="gamma",
        verbose=-1,
    )

    # accuracy is a classification metric; use gamma deviance for a gamma regressor
    # (sklearn returns it negated, so higher is better -> direction="maximize")
    score = cross_val_score(
        reg, X_train_sev, y_train_sev, cv=3, scoring="neg_mean_gamma_deviance"
    ).mean()
    return score


In [8]:

study_sev = optuna.create_study(direction="maximize")
study_sev.optimize(objective_sev, n_trials=5)

print("Best hyperparameters:", study_sev.best_params)
print("Best neg gamma deviance:", study_sev.best_value)

[I 2026-11-07 19:54:12,634] A new study created in memory with name: no-name-c359ad7f-b69d-4436-ba0e-844a2cdf401d
[I 2026-11-07 19:54:12,912] Trial 0 finished with value: -1.8132096797499215 and parameters: {'n_estimators': 35, 'max_depth': 2, 'min_child_samples': 71}. Best is trial 0 with value: -1.8132096797499215.
[I 2026-11-07 19:54:15,119] Trial 1 finished with value: -8996.448588379975 and parameters: {'n_estimators': 189, 'max_depth': 23, 'min_child_samples': 7}. Best is trial 0 with value: -1.8132096797499215.
[I 2026-11-07 19:54:15,711] Trial 2 finished with value: -1.9620334498317071 and parameters: {'n_estimators': 46, 'max_depth': 30, 'min_child_samples': 41}. Best is trial 0 with value: -1.8132096797499215.
[I 2026-11-07 19:54:15,998] Trial 3 finished with value: -1.799262067116347 and parameters: {'n_estimators': 66, 'max_depth': 4, 'min_child_samples': 60}. Best is trial 3 with value: -1.799262067116347.
[I 2026-11-07 19:54:17,096] Trial 4 finished with value: -2.0257557

Best hyperparameters: {'n_estimators': 66, 'max_depth': 4, 'min_child_samples': 60}
Best neg gamma deviance: -1.799262067116347
